# NeoStats — Credit Risk Platform
**Home Credit Default Risk Dataset**

Walk through of the five key areas of the project:
1. Dataset overview & data quality
2. Target distribution & class imbalance
3. Demographic & financial feature analysis
4. External credit scores vs default rate
5. Key business insights


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0a0d14',
    'axes.facecolor':   '#111827',
    'axes.edgecolor':   '#1e2d45',
    'axes.labelcolor':  '#64748b',
    'xtick.color':      '#e2e8f0',
    'ytick.color':      '#e2e8f0',
    'text.color':       '#e2e8f0',
    'grid.color':       '#1e2d45',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
})

ACCENT = '#3b82f6'
GREEN  = '#22c55e'
RED    = '#ef4444'
AMBER  = '#f59e0b'

print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
from src.data.loader import load_and_join

# sample=50000 for quick data exploration
df = load_and_join(sample=50_000)
print(f'Shape: {df.shape}')
df.head(3)

## 2. Dataset Summary & Data Quality

In [ ]:
print('=== Basic Info ===')
print(f'Total rows   : {len(df):,}')
print(f'Total columns: {df.shape[1]}')
print(f'Default rate : {df.TARGET.mean()*100:.2f}%')
print(f'\nDtypes:')
print(df.dtypes.value_counts())
print(f'\nTop 15 columns by missing %:')
print((df.isnull().mean()*100).sort_values(ascending=False).head(15).round(2))

In [ ]:
# Missing value bar chart
miss = (df.isnull().mean()*100).sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(12, 5))
colors = [RED if v > 40 else AMBER if v > 20 else ACCENT for v in miss.values]
ax.barh(miss.index[::-1], miss.values[::-1], color=colors[::-1], edgecolor='none')
ax.set_xlabel('Missing %')
ax.set_title('Top 20 Columns by Missing Value Rate', fontsize=13, fontweight='bold')
ax.axvline(30, color=RED, linestyle='--', alpha=0.5, label='>30% threshold')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart
counts = df.TARGET.value_counts()
bars = axes[0].bar(['Repaid (0)', 'Defaulted (1)'], counts.values,
                   color=[GREEN, RED], width=0.5, edgecolor='none')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01,
                 f'{bar.get_height():,}', ha='center', fontsize=11)
axes[0].set_title('Class Distribution', fontweight='bold')

# Pie
axes[1].pie(counts.values, labels=['Repaid', 'Defaulted'],
            colors=[GREEN, RED], autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor':'#0a0d14','linewidth':2})
axes[1].set_title('Class Proportion', fontweight='bold')

plt.suptitle('⚠ Significant Class Imbalance (~8-9% default rate)', color=AMBER, fontsize=11)
plt.tight_layout()
plt.show()

## 4. Demographic Analysis

In [ ]:
df['AGE'] = -df['DAYS_BIRTH'] / 365
df['EMP_YEARS'] = (-df['DAYS_EMPLOYED'] / 365).clip(lower=0).where(df['DAYS_EMPLOYED'] != 365243)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Target by age
for t, col, lbl in [(0, GREEN, 'Repaid'), (1, RED, 'Defaulted')]:
    axes[0,0].hist(df[df.TARGET==t]['AGE'].dropna(), bins=30, alpha=0.7, color=col, label=lbl, edgecolor='none')
axes[0,0].set_title('Age Distribution by Default Status', fontweight='bold')
axes[0,0].legend()
axes[0,0].set_xlabel('Age (years)')

# Default rate by age group
df['age_group'] = pd.cut(df['AGE'], bins=[18,25,30,35,40,50,60,100],
                          labels=['18-25','25-30','30-35','35-40','40-50','50-60','60+'])
age_dr = df.groupby('age_group', observed=True)['TARGET'].mean()*100
axes[0,1].bar(age_dr.index.astype(str), age_dr.values,
              color=[RED if v>10 else AMBER if v>8 else GREEN for v in age_dr.values],
              edgecolor='none')
axes[0,1].set_title('Default Rate by Age Group', fontweight='bold')
axes[0,1].set_ylabel('Default Rate (%)')

# Income
for t, col, lbl in [(0, GREEN, 'Repaid'), (1, RED, 'Defaulted')]:
    inc = df[df.TARGET==t]['AMT_INCOME_TOTAL'].dropna()
    inc = inc[inc < inc.quantile(.99)]
    axes[1,0].hist(inc, bins=50, alpha=0.7, color=col, label=lbl, edgecolor='none')
axes[1,0].set_xscale('log')
axes[1,0].set_title('Income Distribution (log scale)', fontweight='bold')
axes[1,0].legend()
axes[1,0].set_xlabel('Annual Income')

# Gender default rate
gdr = df.groupby('CODE_GENDER')['TARGET'].agg(['mean','count'])
axes[1,1].bar(gdr.index, gdr['mean']*100, color=[ACCENT,'#06b6d4','#64748b'][:len(gdr)], edgecolor='none')
axes[1,1].set_title('Default Rate by Gender', fontweight='bold')
axes[1,1].set_ylabel('Default Rate (%)')
for i, (idx, row) in enumerate(gdr.iterrows()):
    axes[1,1].text(i, row['mean']*100+0.2, f'{row["mean"]*100:.1f}%', ha='center', fontsize=10)

plt.tight_layout(pad=2)
plt.show()

## 5. External Credit Scores vs Default

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, col in enumerate(['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']):
    bins = np.linspace(0, 1, 11)
    df[f'{col}_bin'] = pd.cut(df[col], bins=bins)
    agg = df.groupby(f'{col}_bin', observed=True)['TARGET'].agg(['mean','count']).dropna()
    x = range(len(agg))
    axes[i].bar(x, agg['mean']*100,
                color=[RED if v>0.15 else AMBER if v>0.10 else GREEN for v in agg['mean']],
                edgecolor='none', width=0.7)
    axes[i].set_title(f'{col} vs Default Rate', fontweight='bold')
    axes[i].set_ylabel('Default Rate (%)')
    axes[i].set_xticks(list(x))
    axes[i].set_xticklabels([str(b) for b in agg.index], rotation=45, ha='right', fontsize=7)

plt.suptitle('External Scores — Strong Inverse Correlation with Default', fontsize=11, color=ACCENT)
plt.tight_layout()
plt.show()

## 6. Financial Feature Analysis

In [ ]:
df['CREDIT_TO_INCOME'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL'].replace(0, np.nan)
df['ANNUITY_TO_INCOME'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL'].replace(0, np.nan)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Credit to income
cti = pd.cut(df['CREDIT_TO_INCOME'].clip(0,8), bins=[0,1,2,3,5,8],
             labels=['<1x','1-2x','2-3x','3-5x','>5x'])
cti_dr = df.groupby(cti, observed=True)['TARGET'].mean()*100
axes[0].bar(cti_dr.index.astype(str), cti_dr.values,
            color=[GREEN,AMBER,AMBER,RED,RED], edgecolor='none')
axes[0].set_title('Default Rate by Credit/Income Ratio', fontweight='bold')
axes[0].set_ylabel('Default Rate (%)')

# Annuity to income
ati = pd.cut(df['ANNUITY_TO_INCOME'].clip(0,0.8), bins=[0,.1,.2,.3,.4,.6,.8],
             labels=['<10%','10-20%','20-30%','30-40%','40-60%','>60%'])
ati_dr = df.groupby(ati, observed=True)['TARGET'].mean()*100
axes[1].bar(ati_dr.index.astype(str), ati_dr.values,
            color=[GREEN,GREEN,AMBER,AMBER,RED,RED], edgecolor='none')
axes[1].set_title('Default Rate by Annuity/Income Ratio', fontweight='bold')
axes[1].set_ylabel('Default Rate (%)')

# Education type
edu = df.groupby('NAME_EDUCATION_TYPE')['TARGET'].mean().sort_values()*100
axes[2].barh(edu.index, edu.values,
             color=[RED if v>10 else AMBER if v>8 else GREEN for v in edu.values],
             edgecolor='none')
axes[2].set_title('Default Rate by Education', fontweight='bold')
axes[2].set_xlabel('Default Rate (%)')

plt.tight_layout()
plt.show()

## 7. Key Business Insights Summary

In [ ]:
insights = [
    ('1', 'Class Imbalance',
     f"Dataset has {df.TARGET.mean()*100:.1f}% default rate. "
     "Requires class-weight balancing and threshold optimisation."),
    ('2', 'Age is a Strong Risk Proxy',
     f"Under-30 applicants default at {df[df.AGE<30].TARGET.mean()*100:.1f}% vs "
     f"{df[df.AGE>=50].TARGET.mean()*100:.1f}% for 50+. Age → financial maturity signal."),
    ('3', 'External Scores are Critical Features',
     "EXT_SOURCE_2 < 0.35 predicts >20% default. Top SHAP contributor across all models."),
    ('4', 'Over-Leverage is High Risk',
     f"Credit/income > 3× doubles default risk. Annuity > 40% of income is unsustainable."),
    ('5', 'Missing Data Carries Signal',
     f"Applicants missing EXT_SOURCE_1 default at "
     f"{df[df.EXT_SOURCE_1.isna()].TARGET.mean()*100:.1f}% vs "
     f"{df[df.EXT_SOURCE_1.notna()].TARGET.mean()*100:.1f}% where present."),
]

print('='*60)
print('  KEY BUSINESS INSIGHTS — NeoStats Credit Risk EDA')
print('='*60)
for num, title, desc in insights:
    print(f'\n  Insight {num}: {title}')
    print(f'  {desc}')
print('\n' + '='*60)